In [0]:
# Imports 
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = SparkSession.builder.appName("BronzeToSilverDelta").getOrCreate()

In [0]:
# Paths

BASE_PATH = "abfss://lakehouse@staqilakehouseen.dfs.core.windows.net"

BRONZE_RAW_DELTA_PATH = f"{BASE_PATH}/bronze_delta/air_quality_raw/"
BRONZE_STRUCTURED_DELTA_PATH = f"{BASE_PATH}/bronze_delta/air_quality_structured/"
SILVER_DELTA_PATH = f"{BASE_PATH}/silver_delta/air_quality/"
GOLD_DELTA_BASE_PATH = f"{BASE_PATH}/gold_delta/"
CHECKPOINT_DELTA_BASE_PATH = f"{BASE_PATH}/checkpoints_delta/"

In [0]:
# Structured data

structured_df = spark.read.format("delta").load(BRONZE_STRUCTURED_DELTA_PATH)

structured_df.printSchema()

root
 |-- source: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- city_id: integer (nullable = true)
 |-- city: string (nullable = true)
 |-- country: string (nullable = true)
 |-- measurement_timestamp: string (nullable = true)
 |-- european_aqi: integer (nullable = true)
 |-- pm10: double (nullable = true)
 |-- pm2_5: double (nullable = true)
 |-- nitrogen_dioxide: double (nullable = true)
 |-- ingestion_timestamp_utc: string (nullable = true)



In [0]:
# Cast Timestamps

structured_df = structured_df.select(
    F.col("source"),
    F.col("latitude"),
    F.col("longitude"),
    F.col("city_id"),
    F.col("city"),
    F.col("country"),
    F.to_timestamp(F.col("measurement_timestamp")).alias("measurement_timestamp"),
    F.col("european_aqi"),
    F.col("pm10"),
    F.col("pm2_5"),
    F.col("nitrogen_dioxide"),
    F.to_timestamp(F.col("ingestion_timestamp_utc")).alias("ingestion_timestamp_utc")
)

# Validate key values

filtered_df = structured_df.filter(
    (F.col("source") == "openmeteo") &
    (F.col("city_id").isNotNull()) &
    (F.col("measurement_timestamp").isNotNull()) &
    (F.col("european_aqi").isNotNull())
)

# Flag data

flagged_df = filtered_df.withColumns({
    "is_valid_aqi": (F.col("european_aqi").isNotNull()) & (F.col("european_aqi") >= 0),
    "is_valid_pm10": (F.col("pm10").isNotNull()) & (F.col("pm10") >= 0),
    "is_valid_pm2_5": (F.col("pm2_5").isNotNull()) & (F.col("pm2_5") >= 0),
    "is_valid_nitrogen_dioxide": (F.col("nitrogen_dioxide").isNotNull()) & (F.col("nitrogen_dioxide") >= 0),
    "is_not_future_measurement": F.col("measurement_timestamp") <= F.current_timestamp()
}
)

silver_df = flagged_df.filter(
    F.col("is_valid_aqi") &
    F.col("is_valid_pm10") &
    F.col("is_valid_pm2_5") &
    F.col("is_valid_nitrogen_dioxide") &
    F.col("is_not_future_measurement")
)

# Duduplication

dedupe_window = Window.partitionBy(
    "city_id",
    "measurement_timestamp"    
    ).orderBy(
        F.col("ingestion_timestamp_utc").desc()
    )

silver_df = silver_df.withColumn(
    "rn",
    F.row_number().over(dedupe_window)
).filter(
    F.col("rn") == 1
).drop(
    "rn"
)

In [0]:
silver_df.show(20, truncate=False)

+---------+--------+---------+-------+------+-------+---------------------+------------+----+-----+----------------+--------------------------+------------+-------------+--------------+-------------------------+-------------------------+
|source   |latitude|longitude|city_id|city  |country|measurement_timestamp|european_aqi|pm10|pm2_5|nitrogen_dioxide|ingestion_timestamp_utc   |is_valid_aqi|is_valid_pm10|is_valid_pm2_5|is_valid_nitrogen_dioxide|is_not_future_measurement|
+---------+--------+---------+-------+------+-------+---------------------+------------+----+-----+----------------+--------------------------+------------+-------------+--------------+-------------------------+-------------------------+
|openmeteo|37.9838 |23.7275  |1      |Athens|Greece |2026-07-09 00:00:00  |30          |22.7|15.3 |42.1            |2026-07-09 20:58:21.635676|true        |true         |true          |true                     |true                     |
|openmeteo|37.9838 |23.7275  |1      |Athens|Gre

In [0]:
# Write to Delta

silver_df.write.format("delta").mode("overwrite").save(SILVER_DELTA_PATH)

In [0]:
# Test Read

spark.read.format("delta").load(SILVER_DELTA_PATH).show(20, truncate=False)

+---------+--------+---------+-------+------+-------+---------------------+------------+----+-----+----------------+--------------------------+------------+-------------+--------------+-------------------------+-------------------------+
|source   |latitude|longitude|city_id|city  |country|measurement_timestamp|european_aqi|pm10|pm2_5|nitrogen_dioxide|ingestion_timestamp_utc   |is_valid_aqi|is_valid_pm10|is_valid_pm2_5|is_valid_nitrogen_dioxide|is_not_future_measurement|
+---------+--------+---------+-------+------+-------+---------------------+------------+----+-----+----------------+--------------------------+------------+-------------+--------------+-------------------------+-------------------------+
|openmeteo|37.9838 |23.7275  |1      |Athens|Greece |2026-07-09 00:00:00  |30          |22.7|15.3 |42.1            |2026-07-09 20:58:21.635676|true        |true         |true          |true                     |true                     |
|openmeteo|37.9838 |23.7275  |1      |Athens|Gre